# FabFlow — Agent-Powered Business Assistant
**EECE 503P/798S • C3**

Run cells top to bottom from this directory. The shared `agent.py` file is the full implementation used by this notebook and `app.py`. The two tool functions actually persist logs when the model calls them.

In [ ]:
%pip install -r requirements.txt

## 1. Business identity and knowledge (PDF + TXT)

In [ ]:
from pathlib import Path
from pypdf import PdfReader
root = Path.cwd()
print((root / "me/business_summary.txt").read_text()[:1800])
print("PDF pages:", len(PdfReader(str(root / "me/about_business.pdf")).pages))

## 2. Persona and instructions — separate labeled blocks

In [ ]:
from agent import PERSONA, INSTRUCTIONS, SYSTEM_PROMPT
print("PERSONA:\n", PERSONA)
print("\nINSTRUCTIONS:\n", INSTRUCTIONS)
print("\nSystem prompt size:", len(SYSTEM_PROMPT), "characters")

## 3. MCP-style schemas and executable tools

In [ ]:
from agent import TOOL_CATALOG, TOOLS, record_customer_interest, record_feedback
import json
print(json.dumps(TOOL_CATALOG, indent=2))
# Tools are executed by the agent after the LLM requests a tool call.
# Call functions directly only if you intentionally want to write local logs.

## 4. Router (three intents) and short-term memory

In [ ]:
from agent import classify_intent, _normalize_history, run_agent
# Requires OPENAI_API_KEY in the untracked .env file
examples = ["What materials do you print?", "Can I get a quote?", "I wish your hours were longer"]
for example in examples:
    print(repr(example), "=>", classify_intent(example))

## 5. Multi-turn demonstration (tool calls and reflection)
`run_agent` passes model tools and prior conversation history. The console shows `[ROUTER]`, `[TOOL]`, and `[REFLECTION]` logs.

In [ ]:
history = []
for user_msg in ["I need a quote for 10 PLA prototypes.", "I am Sam Lee; my email is sam@example.com"]:
    reply = run_agent(user_msg, history)
    print("USER:", user_msg, "\nNOVA:", reply, "\n")
    history.extend([{"role": "user", "content": user_msg}, {"role": "assistant", "content": reply}])

## 6. Launch Gradio chatbot

In [ ]:
import gradio as gr
demo = gr.ChatInterface(fn=run_agent, title="FabFlow | Nova", description="Ask about services, request a quote or share feedback.", examples=["What services are available?", "I need a print quote", "What is the nozzle diameter?"])
demo.launch()

## 7. Bonus reflection
Reflection is enabled by default (`ENABLE_REFLECTION=true`). Each response prints its original draft and revision. To disable: `run_agent("Hi", use_reflection=False)`.

In [ ]:
from agent import reflect_response, REFLECTION_ENABLED
print("Reflection enabled:", REFLECTION_ENABLED)